# WWW::HorizonsEphemerisSystem basic usage

Anton Antonov  
April 2026

---

## Setup

In [1]:
use WWW::HorizonsEphemerisSystem;

----

## Basic usage

Cartesian position of Mars with default parameters.

In [2]:
my $mars-default = horizons-ephemeris-data(
    'state',
    '499',
    'position',
    :modifier<data>,
);

[{Calendar Date (TDB) => A.D. 2026-Apr-12 19:48:00.0000, JDTDB => 2461143.325000000, VX => 6.296302421498797E+00, VY => 2.570383052576669E+01, VZ => 3.842775568724122E-01, X => 2.014122495284725E+08, Y => -4.654592594728635E+07, Z => -5.888122905150859E+06}]

Cartesian position of Mars for a specific date.

In [3]:
my $mars-at-date = horizons-ephemeris-data(
    'state',
    ['499', {
        dates => '2026-Apr-12 00:00:00',
    }],
    'position',
    :modifier<data>,
);

[{Calendar Date (TDB) => A.D. 2026-Apr-12 00:00:00.0000, JDTDB => 2461142.500000000, VX => 6.511407089780921E+00, VY => 2.565403899986164E+01, VZ => 3.779598135296425E-01, X => 2.009557804368319E+08, Y => -4.837633215147470E+07, Z => -5.915289234365197E+06}]

Azimuth/Elevation of the Moon from Moscow (lat, lon) for next 12 hours.

In [4]:
my @dates = do with DateTime.now { ($_, $_.later(:12hours), $_.later(minutes => 12*60 + 10))};
@dates = @dates».Str.map({ $_.substr(^19).subst('T', ' ') })

[2026-04-12 15:48:02 2026-04-13 03:48:02 2026-04-13 03:58:02]

In [7]:
my $tycho-from-moscow = horizons-ephemeris-data(
    'observer',
    {
        target => '301',
        center => (55.7505, 37.6175),
        :@dates,
    },
    <azimuth elevation>,
    :modifier<data>,
);

[{Azi_(a-app) => *, Date_________JDUT => 2461143.158356482, Elev_(a-app) => } {Azi_(a-app) => *, Date_________JDUT => 2461143.658356482, Elev_(a-app) => m} {Azi_(a-app) => *, Date_________JDUT => 2461143.665300926, Elev_(a-app) => m}]

Here is a range of dates from now to 12 hours from now with step 10 min:

In [8]:
my $start = DateTime.now;
my $end = $start.later(:12hours);
my $current = $start;
my @dates;

while $current <= $end {
    @dates.push($current);
    $current = $current.later(:30minutes);
}

@dates = @dates».Str.map({ $_.substr(^19).subst('T', ' ') });

deduce-type(@dates);

Vector(Atom((Str)), 25)

Orbital elements of Saturn moon Pandora for a specific list of date-times.

In [9]:
my $pandora-elements = horizons-ephemeris-data(
    'orbital-elements',
    ['617', {
        center => '699',
        :@dates
    }],
    'all',
    :modifier<association>,
);

deduce-type($pandora-elements)

Assoc(Atom((Str)), Assoc(Atom((Str)), Atom((Str)), 14), 25)

In [17]:
#%html
$pandora-elements.values
==> to-html(field-names => ['Date', |$pandora-elements.values.head.keys.grep({ $_ ∉ ['Calendar Date (TDB)', 'Date'] }).sort])

Date,ApoapsisDistance,AscendingNodeLongitude,Eccentricity,Inclination,MeanAnomaly,MeanMotion,OrbitalPeriod,PeriapsisDate,PeriapsisDistance,PerifocusArgument,SemiMajorAxis,TrueAnomaly
2461143.491736111,1.435975464862458E+05,1.696266068163932E+02,8.704200954127541E-03,2.803814083861474E+01,3.546313786575990E+02,6.569709619240316E-03,5.479694246237147E+04,2.461143501194188E+06,1.411193137199825E+05,2.816432144345980E+02,1.423584301031141E+05,3.545370347460589E+02
2461143.450069444,1.435437364082274E+05,1.696264629329557E+02,8.333337426722690E-03,2.803813131119229E+01,3.432188942903235E+02,6.569779268771062E-03,5.479636153245394E+04,2.461143479632958E+06,1.411711115101257E+05,2.691885756307788E+02,1.423574239591766E+05,3.429404155197580E+02
2461143.262569444,1.427526506076240E+05,1.696260524868080E+02,2.836059604610216E-03,2.803752382414216E+01,2.937169885172606E+02,6.570366576651716E-03,5.479146342904012E+04,2.461143379330735E+06,1.419452304476451E+05,2.112946575601258E+02,1.423489405276345E+05,2.934190239562113E+02
2461143.304236111,1.429855890774778E+05,1.696259355771318E+02,4.461493023380862E-03,2.803765495479096E+01,3.038226594315620E+02,6.570259068610052E-03,5.479235997252061E+04,2.461143403197308E+06,1.417153976116147E+05,2.250537670411627E+02,1.423504933445463E+05,3.033966131361567E+02
2461143.241736111,1.426301071595618E+05,1.696261223835750E+02,1.978934025217020E-03,2.803748588302776E+01,2.898233020886270E+02,6.570403373793266E-03,5.479115657280605E+04,2.461143365355660E+06,1.420667109393548E+05,2.032561352643696E+02,1.423484090494583E+05,2.896097920209489E+02
2461143.533402778,1.435961023412441E+05,1.696265899481182E+02,8.693514592768114E-03,2.803814638125637E+01,6.046860581981134E+00,6.569704324765179E-03,5.479698662281388E+04,2.461143522749816E+06,1.411209108324096E+05,2.940950659106069E+02,1.423585065868269E+05,6.152948326270589E+00
2461143.554236111,1.435745320606048E+05,1.696265321411311E+02,8.544363971529620E-03,2.803816672033216E+01,1.175520570678655E+01,6.569727484413700E-03,5.479679345209969E+04,2.461143533526634E+06,1.411418119884068E+05,3.003203734605294E+02,1.423581720245058E+05,1.195678740369957E+01
2461143.179236111,1.424528714390675E+05,1.696261949528764E+02,7.371962278711392E-04,2.803746024948896E+01,6.716740583604522E+01,6.570436328071055E-03,5.479088176563893E+04,2.461143060918165E+06,1.422429947204443E+05,3.011588147222383E+01,1.423479330797559E+05,6.724529076656415E+01
2461143.658402778,1.432748685418470E+05,1.696263415348483E+02,6.471724096973495E-03,2.803851946005737E+01,4.022912636910011E+01,6.570044312613364E-03,5.479415097838251E+04,2.461143587533410E+06,1.414323221552311E+05,3.315130704634129E+02,1.423535953485390E+05,4.071105898798860E+01
2461143.616736111,1.434303887338300E+05,1.696263383230700E+02,7.548238149121793E-03,2.803833947688337E+01,2.887049654926934E+01,6.569888029726037E-03,5.479545440822557E+04,2.461143565875387E+06,1.412813169752949E+05,3.190055055386144E+02,1.423558528545625E+05,2.929160312393773E+01


Using alternative date range spec:

In [11]:
my $date-range = "{@dates.head} + {@dates.tail} + 10 m"

2026-04-12 15:48:06 + 2026-04-13 03:48:06 + 10 m